In [ ]:
!pip install ijson boto3 --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 149.7/149.7 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.5/140.5 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.0/15.0 MB 50.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 4.9 MB/s eta 0:00:00


In [ ]:
import boto3
import ijson
import torch
import pandas as pd
import numpy as np
import torch.nn as nn
from tqdm import tqdm
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from google.colab import userdata

In [ ]:
def parsing_json_s3_all():
  ACCESS_KEY = userdata.get("ACCESS_KEY")
  SECRET_KEY = userdata.get("SECRET_KEY")

  s3 = boto3.client(
    "s3",
    endpoint_url="https://storage.yandexcloud.net",
    aws_access_key_id=ACCESS_KEY,
    aws_secret_access_key=SECRET_KEY
  )

  response = s3.get_object(Bucket="slovo-mediapipe", Key="slovo_mediapipe.json")
  parser = ijson.kvitems(response["Body"], "")

  counter = 0
  keys, videos, frames, dots = [], [], [], []
  for key, value in parser:
    counter += 1
    if counter%100 == 0:
      print(counter)
    keys.append(key)
    for i in range(len(value)):
      item = value[i]["hand 1"]
      for dot in item:
        dots.append(np.float32(dot["x"]))
        dots.append(np.float32(dot["y"]))
        dots.append(np.float32(dot["z"]))
      frames.append(dots)
      dots = []
    videos.append(frames)
    frames = []
  return videos, keys

In [ ]:
def x_preprocessing(videos: list):
    video_preprocessed, mistakes = [], []
    for i, video in tqdm(enumerate(videos)):
      try:
        video_np = np.array(video)
        A, F = video_np.shape
        old_time = np.linspace(0, 1, A)
        new_time = np.linspace(0, 1, 48)
        resampled_video = np.zeros((48, 63), dtype=video_np.dtype)
        for f in range(F):
            resampled_video[:, f] = np.interp(
                new_time,
                old_time,
                video_np[:, f]
            )
        resampled_video = resampled_video.reshape(48, 21, 3)
        wrist = resampled_video[:, 0:1, :]
        resampled_video_normalized = resampled_video - wrist
        resampled_video_normalized = resampled_video_normalized.reshape(48, 63)
        video_preprocessed.append(resampled_video_normalized)
      except:
          mistakes.append(i)
    return video_preprocessed, mistakes

In [ ]:
def y_preprocessing(keys: list) -> list:
  df = pd.read_csv("annotations.csv", sep="\t")
  attachments, labels = df["attachment_id"].tolist(), df["text"].tolist()
  labels_dict = {item: labels[i] for i, item in enumerate(attachments)}

  labels_classification = [labels_dict[id] for id in keys]
  encoder = LabelEncoder()
  labels_encoded = np.int32(encoder.fit_transform(labels_classification))
  return labels_encoded, encoder.classes_

In [ ]:
def hold_out(X: list, y: list):
  X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, shuffle=True)

  train_ds = GestureDataset(X_train, y_train)
  test_ds = GestureDataset(X_test, y_test)

  train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
  test_loader = DataLoader(test_ds, batch_size=32)
  return train_loader, test_loader

In [ ]:
class GestureRNN(nn.Module):
  def __init__(self,
               input_size=63,
               hidden_size=128,
               num_layers=2,
               num_classes=1000):
    super().__init__()
    self.lstm = nn.GRU(
      input_size=input_size,
      hidden_size=hidden_size,
      num_layers=num_layers,
      batch_first=True,
      bidirectional=True
    )
    # self.lstm = nn.LSTM(
    #   input_size=input_size,
    #   hidden_size=hidden_size,
    #   num_layers=num_layers,
    #   batch_first=True,
    #   bidirectional=True
    # )
    self.fc = nn.Linear(hidden_size*2, num_classes)

  def forward(self, x):
    out, _ = self.lstm(x)
    out = out[:, -1, :]
    out = self.fc(out)
    return out

In [ ]:
class GestureDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

In [ ]:
class GestureCNN(nn.Module):
  def __init__(self, num_classes=1000):
    super().__init__()

    self.conv = nn.Sequential(
      nn.Conv1d(63, 32, kernel_size=3, padding=1),
      nn.ReLU(),
      nn.MaxPool1d(2),

      # nn.Conv1d(32, 64, kernel_size=3, padding=1),
      # nn.ReLU(),
      # nn.MaxPool1d(2),
    )

    self.fc = nn.Sequential(
      nn.Flatten(),
      nn.Linear(64 * 12, 128),
      nn.ReLU(),
      nn.Linear(128, num_classes)
    )

  def forward(self, x):
    x = x.permute(0, 2, 1)
    x = self.conv(x)
    x = self.fc(x)
    return x

In [ ]:
corpus, keys = parsing_json_s3_all()

In [ ]:
X, mistakes = x_preprocessing(videos=corpus)

20000it [00:10, 1846.19it/s]


In [ ]:
keys_preprocessed = [key for i, key in tqdm(enumerate(keys)) if i not in mistakes]
y, classes = y_preprocessing(keys=keys_preprocessed)

20000it [00:00, 1956709.20it/s]


In [ ]:
train_loader, test_loader = hold_out(X=X, y=y)

/tmp/ipykernel_48451/123938772.py:3: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:253.)
  self.X = torch.tensor(X, dtype=torch.float32)


In [ ]:
print(len(X))
print(len(y))
print()
print(len(X[0]))
print(len(X[0][0]))
print()
print(X[0])
print(y[0])

19990
19990

48
63

[[ 0.          0.          0.         ...  0.125      -0.05499995
  -0.06      ]
 [ 0.          0.          0.         ...  0.1513617  -0.05659568
  -0.04919149]
 [ 0.          0.          0.         ...  0.15761703 -0.05580845
  -0.04610638]
 ...
 [ 0.          0.          0.         ...  0.12080851 -0.06800002
  -0.05438298]
 [ 0.          0.          0.         ...  0.12159574 -0.06840423
  -0.058     ]
 [ 0.          0.          0.         ...  0.11499998 -0.07099998
  -0.053     ]]
4


In [ ]:
# model_rnn = Gesture/RNN()
model_cnn = GestureCNN()

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model_cnn.parameters(), lr=1e-3)

In [ ]:
for epoch in range(200):
  for x_train_loader, y_train_loader in train_loader:
    optimizer.zero_grad()
    outputs = model_cnn(x_train_loader)
    loss = criterion(outputs, y_train_loader)
    loss.backward()
    optimizer.step()

  correct, total = 0, 0
  with torch.no_grad():
    for x_test_loader, y_test_loader in test_loader:
      outputs = model_cnn(x_test_loader)
      _, predicted = torch.max(outputs, 1)
      total += y_test_loader.size(0)
      correct += (predicted == y_test_loader).sum().item()
  print(f"Epoch {epoch}. Loss {loss.item()}")
  print(f"Epoch {epoch}. Accuracy {correct/total}")

Epoch 0. Loss 1.1923202276229858
Epoch 0. Accuracy 0.2941470735367684
Epoch 1. Loss 0.50252765417099
Epoch 1. Accuracy 0.288144072036018
Epoch 2. Loss 0.15679329633712769
Epoch 2. Accuracy 0.2891445722861431
Epoch 3. Loss 0.44040536880493164
Epoch 3. Accuracy 0.29239619809904954
Epoch 4. Loss 0.5127739310264587
Epoch 4. Accuracy 0.29514757378689344
Epoch 5. Loss 0.32502833008766174
Epoch 5. Accuracy 0.2896448224112056
Epoch 6. Loss 0.33715569972991943
Epoch 6. Accuracy 0.2968984492246123
Epoch 7. Loss 0.34204116463661194
Epoch 7. Accuracy 0.3024012006003001
Epoch 8. Loss 0.7046524882316589
Epoch 8. Accuracy 0.2876438219109555
Epoch 9. Loss 0.3718166649341583
Epoch 9. Accuracy 0.28789394697348675
Epoch 10. Loss 0.4263324737548828
Epoch 10. Accuracy 0.295647823911956
Epoch 11. Loss 0.3820742666721344
Epoch 11. Accuracy 0.2876438219109555
Epoch 12. Loss 0.6915375590324402
Epoch 12. Accuracy 0.2873936968484242
Epoch 13. Loss 0.4017689526081085
Epoch 13. Accuracy 0.29164582291145574
Epoch 1

In [ ]:
model_cnn.eval()

correct = 0
total = 0

with torch.no_grad():
    for x_test_loader, y_test_loader in test_loader:
        outputs = model_cnn(x_test_loader)

        _, predicted = torch.max(outputs, 1)

        total += y_test_loader.size(0)
        correct += (predicted == y_test_loader).sum().item()

accuracy = correct / total
print(f"Accuracy: {accuracy}")

Accuracy: 0.28264132066033015
